In [1]:
"""
mo_check_imputation_amounts.ipynb

This script is used to check the imputation amounts

authors: Roy Oelen

"""

'\nmo_check_imputation_amounts.ipynb\n\nThis script is used to check the imputation amounts\n\nauthors: Roy Oelen\n\n'

In [13]:
###########
# imports #
###########

# object
from pycisTopic.cistopic_class import CistopicObject
import joblib
import numpy as np
from scipy import sparse
import hashlib
import pandas as pd
import pickle


In [3]:
#############
# functions #
#############

# method to create md5
def create_md5_file(input_file):
    """
    Creates an MD5 hash of the specified file and writes it to a new file with the same name but .md5 added to the extension.

    Args:
        input_file (str): The path to the input file for which the MD5 hash should be created.

    Returns:
        int: Returns 0 on success, 1 on failure.

    Raises:
        FileNotFoundError: Thrown if the input file does not exist.
        IOError: Thrown if there is an error reading the input file (like permission denied) or writing the output md5 file.
    """
    try:
        # get an md5 of the file
        digest = None
        with open(input_file, "rb") as f:
            # try Python 3.11+ method if it is available
            if callable(getattr(hashlib, 'file_digest', None)):
                # digest with one command
                digest = hashlib.file_digest(f, 'md5')
            # or the older 3.8+ method if we don't have the newer method
            else:
                # initialize digest
                digest = hashlib.md5()
                # read file in chunks
                while chunk := f.read(8192):
                    # update digestion
                    digest.update(chunk)       
        # get the output path of the md5
        output_md5_loc = ''.join([input_file, '.md5'])
        # and write that
        with open(output_md5_loc, "w") as m:
            m.write(digest.hexdigest())
        # upon success, return 0
        return 0
    except Exception as e:
        print(f"Exception occured upon md5 file creation: {e}")
        return 1


In [4]:
###########################
# load imputation results #
###########################

# location to store the object
pycistopic_object_wimputations_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/objects/merged_major_and_minor_celltypes_120topics_imputed.joblib'

# initalize the variable
cistopic_obj = None

# load the object
cistopic_obj = joblib.load(pycistopic_object_wimputations_loc)


In [5]:
##############################
# get non-zero accessibility #
##############################

# for ease, make reference to the imputed accessibility matrix
mtx = cistopic_obj.mtx

# get the fraction of non-zeroes per region (column) across all cells (rows)
nonzero_counts_mtx = np.count_nonzero(mtx, axis=1)


In [10]:
# get the number of cells
n_cells_mtx = mtx.shape[1]
# calculate the fraction of non-zeroes per region
fraction_nonzero_per_region_mtx = nonzero_counts_mtx / n_cells_mtx


In [11]:
#############################
# get average accessibility #
#############################

# Calculate average accessibility per region (column) across all cells (rows)
average_accessibility_per_region = np.mean(mtx, axis=1)


In [12]:
################################
# make into a pandas dataframe #
################################

# create a dataframe from the fraction nonzero values
df_fraction_nonzero_average = pd.DataFrame({'region': cistopic_obj.feature_names, 'fraction_nonzero': fraction_nonzero_per_region_mtx , 'average_accessibility': average_accessibility_per_region})
# add the region names as index
df_fraction_nonzero_average.index = df_fraction_nonzero_average['region']

# set the location of the dataframe
df_fraction_nonzero_average_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/tables/merged_major_and_minor_celltypes_120topics_imputed_fraction_nonzero.tsv.gz'
# save the dataframe
df_fraction_nonzero_average.to_csv(df_fraction_nonzero_average_loc, sep='\t', index=False, compression='gzip', header=True)
# make an md5 of the file
create_md5_file(df_fraction_nonzero_average_loc)


0

In [14]:
############################
# load non-imputed results #
############################

# clear memory
cistopic_obj = None
# set location of non-imputed object
pycistopic_object_no_imputations_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/objects/merged_major_and_minor_celltypes_120topics.pkl'
# load the object
with open(pycistopic_object_no_imputations_loc, 'rb') as f:
    cistopic_obj = pickle.load(f)


In [30]:
##############################
# get non-zero accessibility #
##############################

# for ease, make reference to the non-imputed accessibility matrix
frag_mtx = cistopic_obj.fragment_matrix

# get the mean for dimension 0 (rows) in the sparse matrix (as 1D array)
mean_frag_mtx_dim0 = frag_mtx.mean(axis=1).A1


In [ ]:
# get the fraction of non-zeroes per region (column) across all cells (rows)
nonzero_counts_frag_mtx = np.array((frag_mtx != 0).sum(axis=1)).ravel()


In [36]:
# and calculate the fraction
n_cells_frag_mtx = frag_mtx.shape[1]
fraction_nonzero_per_region_frag_mtx = nonzero_counts_frag_mtx / n_cells_frag_mtx

In [38]:
################################
# make into a pandas dataframe #
################################

# create a dataframe from the fraction nonzero values
df_fraction_nonzero_average_nonimp = pd.DataFrame({'region': cistopic_obj.region_data.index, 'fraction_nonzero': fraction_nonzero_per_region_frag_mtx , 'average_accessibility': mean_frag_mtx_dim0})
# add the region names as index
df_fraction_nonzero_average_nonimp.index = df_fraction_nonzero_average_nonimp['region']

# set the location of the dataframe
df_fraction_nonzero_average_nonimp_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/tables/merged_major_and_minor_celltypes_120topics_nonimputed_fraction_nonzero.tsv.gz'
# save the dataframe
df_fraction_nonzero_average_nonimp.to_csv(df_fraction_nonzero_average_nonimp_loc, sep='\t', index=False, compression='gzip', header=True)
# make an md5 of the file
create_md5_file(df_fraction_nonzero_average_nonimp_loc)


0